In [5]:
import cv2
import numpy as np
import time

from collections import Counter

from tensorflow.keras.models import load_model
from tensorflow.keras.applications.resnet50 import preprocess_input

In [6]:
# ============================================================
# FACIAL EXPRESSION MODEL CONFIGURATION
# ============================================================

MODEL_PATH = "best_resnet50.keras"

CONF_THRESHOLD = 0.50
EMOTION_INTERVAL = 0.2

# Number of recent predictions used for smoothing
SMOOTH_WINDOW = 15


# ============================================================
# FACIAL EXPRESSION CLASSES
# ============================================================

CLASSES = [
    "angry",
    "disgust",
    "fear",
    "happy",
    "neutral",
    "sad",
    "surprise"
]


LABEL_MAP = {
    0: "angry",
    1: "disgust",
    2: "fear",
    3: "happy",
    4: "neutral",
    5: "sad",
    6: "surprise"
}


# ============================================================
# CONFIDENCE CLASSIFICATION
# ============================================================

CONFIDENT_EMOTIONS = {
    "happy",
    "neutral"
}


NOT_CONFIDENT_EMOTIONS = {
    "angry",
    "disgust",
    "fear",
    "sad",
    "surprise"
}


# ============================================================
# LOAD MODEL
# ============================================================

model = load_model(
    MODEL_PATH,
    custom_objects={
        "preprocess_input": preprocess_input
    },
    safe_mode=False
)

print("FACIAL EXPRESSION MODEL LOADED SUCCESSFULLY!")

FACIAL EXPRESSION MODEL LOADED SUCCESSFULLY!


In [7]:
# ============================================================
# CHECK MODEL
# ============================================================

print("Input shape:", model.input_shape)
print("Output shape:", model.output_shape)
print("Number of classes:", model.output_shape[-1])

Input shape: (None, 256, 256, 3)
Output shape: (None, 7)
Number of classes: 7


In [8]:
# ============================================================
# FACE DETECTOR
# ============================================================

face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

print("FACE DETECTOR LOADED SUCCESSFULLY!")

FACE DETECTOR LOADED SUCCESSFULLY!


In [9]:
# ============================================================
# INTERVIEW TRACKING VARIABLES
# ============================================================

# Every accepted emotion prediction
emotion_history = []

# Every confidence classification
confidence_history = []

# Recent emotions for smoothing
recent_emotions = []

# Recent confidence classifications
recent_confidence_classes = []

# Timing
last_emotion_time = 0

# Interview state
interview_running = True

In [10]:
# ============================================================
# PREPROCESS FACE
# ============================================================

def preprocess_face(face):

    # Resize to model input size
    face = cv2.resize(
        face,
        (256, 256)
    )

    # OpenCV uses BGR
    # Convert BGR -> RGB
    face = cv2.cvtColor(
        face,
        cv2.COLOR_BGR2RGB
    )

    # Convert to float32
    face = face.astype(np.float32)

    # ResNet50 preprocessing
    face = preprocess_input(face)

    # Add batch dimension
    face = np.expand_dims(
        face,
        axis=0
    )

    return face

In [11]:
# ============================================================
# PREDICT CURRENT EMOTION
# ============================================================

def predict_emotion(face):

    # Preprocess
    processed_face = preprocess_face(face)

    # Model prediction
    prediction = model.predict(
        processed_face,
        verbose=0
    )[0]

    # Get highest probability
    predicted_class = int(
        np.argmax(prediction)
    )

    confidence = float(
        prediction[predicted_class]
    )

    # Convert number to emotion
    emotion = LABEL_MAP[predicted_class]

    return emotion, confidence, prediction

In [12]:
# ============================================================
# CLASSIFY CONFIDENCE
# ============================================================

def classify_confidence(emotion):

    if emotion in CONFIDENT_EMOTIONS:

        return "confident"

    elif emotion in NOT_CONFIDENT_EMOTIONS:

        return "not_confident"

    else:

        return "unknown"

In [13]:
# ============================================================
# RECORD PREDICTION
# ============================================================

def record_prediction(emotion, confidence):

    global emotion_history
    global confidence_history
    global recent_emotions
    global recent_confidence_classes

    # Ignore low-confidence predictions
    if confidence < CONF_THRESHOLD:

        return "uncertain"

    # Record emotion
    emotion_history.append(emotion)

    # Keep recent emotions
    recent_emotions.append(emotion)

    if len(recent_emotions) > SMOOTH_WINDOW:

        recent_emotions.pop(0)


    # Convert to binary confidence class
    confidence_class = classify_confidence(
        emotion
    )

    # Record confidence class
    confidence_history.append(
        confidence_class
    )

    # Keep recent confidence classes
    recent_confidence_classes.append(
        confidence_class
    )

    if len(recent_confidence_classes) > SMOOTH_WINDOW:

        recent_confidence_classes.pop(0)


    return confidence_class

In [14]:
# ============================================================
# REAL-TIME CONFIDENCE RESULT
# ============================================================

def calculate_realtime_confidence():

    if len(confidence_history) == 0:

        return {
            "confident_percentage": 0.0,
            "not_confident_percentage": 0.0,
            "overall": "unknown"
        }


    counts = Counter(
        confidence_history
    )

    total = len(confidence_history)


    confident_percentage = (
        counts.get("confident", 0)
        / total
    ) * 100


    not_confident_percentage = (
        counts.get("not_confident", 0)
        / total
    ) * 100


    if confident_percentage >= not_confident_percentage:

        overall = "confident"

    else:

        overall = "not_confident"


    return {
        "confident_percentage": confident_percentage,
        "not_confident_percentage": not_confident_percentage,
        "overall": overall
    }

In [15]:
# ============================================================
# FACIAL EXPRESSION DISTRIBUTION
# ============================================================

def calculate_emotion_distribution():

    if len(emotion_history) == 0:

        return {emotion: 0.0 for emotion in CLASSES}


    counts = Counter(
        emotion_history
    )

    total = len(emotion_history)


    percentages = {}

    for emotion in CLASSES:

        percentages[emotion] = (
            counts.get(emotion, 0)
            / total
        ) * 100


    return percentages

In [17]:
# ============================================================
# REAL-TIME INTERVIEW LOOP
# ============================================================

last_emotion_time = 0

current_emotion = "Detecting..."
current_confidence = 0.0
current_confidence_class = "unknown"


while True:

    # --------------------------------------------------------
    # Read camera frame
    # --------------------------------------------------------

    ret, frame = cap.read()

    if not ret:

        print("Could not read camera frame.")

        break


    # --------------------------------------------------------
    # Flip camera for natural display
    # --------------------------------------------------------

    frame = cv2.flip(
        frame,
        1
    )


    # --------------------------------------------------------
    # Convert to grayscale for face detection
    # --------------------------------------------------------

    gray = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2GRAY
    )


    # --------------------------------------------------------
    # Detect faces
    # --------------------------------------------------------

    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(80, 80)
    )


    # --------------------------------------------------------
    # If a face is detected
    # --------------------------------------------------------

    if len(faces) > 0:

        # Take the largest face
        x, y, w, h = max(
            faces,
            key=lambda rect: rect[2] * rect[3]
        )


        # Crop face
        face = frame[
            y:y+h,
            x:x+w
        ]


        # Make sure face is valid
        if face.size > 0:

            current_time = time.time()


            # ------------------------------------------------
            # Run emotion model every EMOTION_INTERVAL
            # ------------------------------------------------

            if (
                current_time - last_emotion_time
                >= EMOTION_INTERVAL
            ):

                last_emotion_time = current_time


                try:

                    # Predict current emotion
                    emotion, confidence, prediction = (
                        predict_emotion(face)
                    )


                    # Record it
                    confidence_class = record_prediction(
                        emotion,
                        confidence
                    )


                    # Only update display if accepted
                    if confidence >= CONF_THRESHOLD:

                        current_emotion = emotion

                        current_confidence = confidence

                        current_confidence_class = (
                            confidence_class
                        )


                except Exception as e:

                    print(
                        "Prediction error:",
                        e
                    )


        # ----------------------------------------------------
        # Draw face rectangle
        # ----------------------------------------------------

        cv2.rectangle(
            frame,
            (x, y),
            (x+w, y+h),
            (255, 255, 255),
            2
        )


    else:

        current_emotion = "No face"


    # ========================================================
    # CALCULATE REAL-TIME OVERALL RESULT
    # ========================================================

    result = calculate_realtime_confidence()


    # ========================================================
    # DISPLAY CURRENT EMOTION
    # ========================================================

    cv2.putText(
        frame,
        f"Current: {current_emotion}",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 255, 255),
        2
    )


    # ========================================================
    # DISPLAY CURRENT MODEL CONFIDENCE
    # ========================================================

    cv2.putText(
        frame,
        f"Model confidence: "
        f"{current_confidence * 100:.1f}%",
        (20, 75),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.65,
        (255, 255, 255),
        2
    )


    # ========================================================
    # DISPLAY CONFIDENT / NOT CONFIDENT
    # ========================================================

    cv2.putText(
        frame,
        f"Current indicator: "
        f"{current_confidence_class}",
        (20, 110),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.65,
        (255, 255, 255),
        2
    )


    # ========================================================
    # DISPLAY OVERALL CONFIDENCE
    # ========================================================

    cv2.putText(
        frame,
        f"Overall: {result['overall']}",
        (20, 155),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 255, 255),
        2
    )


    # ========================================================
    # DISPLAY PERCENTAGES
    # ========================================================

    cv2.putText(
        frame,
        f"Confident: "
        f"{result['confident_percentage']:.1f}%",
        (20, 195),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.65,
        (255, 255, 255),
        2
    )


    cv2.putText(
        frame,
        f"Not Confident: "
        f"{result['not_confident_percentage']:.1f}%",
        (20, 230),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.65,
        (255, 255, 255),
        2
    )


    # ========================================================
    # DISPLAY WINDOW
    # ========================================================

    cv2.imshow(
        "AI Interview - Facial Expression",
        frame
    )


    # ========================================================
    # PRESS Q TO END INTERVIEW
    # ========================================================

    key = cv2.waitKey(1) & 0xFF

    if key == ord("q"):

        break


# ============================================================
# RELEASE CAMERA
# ============================================================

cap.release()

cv2.destroyAllWindows()

In [18]:
# ============================================================
# FINAL INTERVIEW RESULT
# ============================================================

final_confidence = calculate_realtime_confidence()

emotion_distribution = (
    calculate_emotion_distribution()
)


print()
print("=" * 60)
print("              FINAL INTERVIEW RESULT")
print("=" * 60)

print()

print(
    f"Overall classification: "
    f"{final_confidence['overall'].upper()}"
)

print()

print(
    f"Confident: "
    f"{final_confidence['confident_percentage']:.2f}%"
)

print(
    f"Not Confident: "
    f"{final_confidence['not_confident_percentage']:.2f}%"
)

print()

print("-" * 60)
print("FACIAL EXPRESSION DISTRIBUTION")
print("-" * 60)

for emotion in CLASSES:

    print(
        f"{emotion:10s}: "
        f"{emotion_distribution[emotion]:.2f}%"
    )

print()

print(
    f"Total accepted predictions: "
    f"{len(emotion_history)}"
)

print("=" * 60)


              FINAL INTERVIEW RESULT

Overall classification: CONFIDENT

Confident: 71.43%
Not Confident: 28.57%

------------------------------------------------------------
FACIAL EXPRESSION DISTRIBUTION
------------------------------------------------------------
angry     : 24.22%
disgust   : 3.73%
fear      : 0.00%
happy     : 10.56%
neutral   : 60.87%
sad       : 0.62%
surprise  : 0.00%

Total accepted predictions: 161
